**Importing the Dataset**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Steam Game Analysis/games_march2025_full.csv", encoding="latin1", on_bad_lines="skip", engine='python')
# latin1 = handles weird characters
# on_bad_lines='skip' → skips broken rows ("" are borken in this case(missing))

df.head()

**Explore Raw Data**

In [ ]:
print("Shape of the dataset: ",df.shape) # rows, columns
print("Columns of the dataset: \n",df.columns) # column names
print("Data types and missing values: \n",df.info()) # data types & missing values
print("Basic stats: \n",df.describe()) # basic stats
print("Missing values: \n",df.isnull().sum())  # see exactly what’s missing

**Data Cleaning Process**

In [ ]:
df_clean = df.copy()

In [ ]:
# Remove rows with missing key values
df_clean = df_clean.dropna(subset=[
    "price",
    "genres",
    "positive",
    "negative",
    "average_playtime_forever"
]).copy()

In [ ]:
# Remove games with no reviews(prevents 0 devision and 0)
df_clean = df_clean[(df_clean["positive"] + df_clean["negative"]) > 0].copy()

# Creating a REAL rating column instead of having positive and negative (0-1 scale)
df_clean["rating"] = df_clean["positive"] / (df_clean["positive"] + df_clean["negative"])

# Fill NaN with 0 (if there are no reviews, the rating is 0)
#df_clean["rating"] = df_clean["rating"].fillna(0)

# Convert to a 100-point scale for readability
df_clean["rating"] = (df_clean["rating"] * 100).round(1)

df_clean[["name","positive","negative","rating"]].head()
print(df_clean["rating"].describe())

**Visualizing**

In [ ]:
from numpy import astype
import ast

#REMOVE rows with actual missing genre data first (fixes the ['nan'] issue)
df_clean = df_clean.dropna(subset=["genres"])

#Convert to string and immediately clean up brackets/quotes
# fixes the "['nan']" visual error
df_clean["genres"] = (
    df_clean["genres"]
    .astype(str)
    .str.replace(r"[\[\]'\"']", "", regex=True)# Removes [ ] ' "
    .str.replace(r"\\", "", regex=True) # Deletes \

)

#Spliting genres into lists (From this "Action, RPG" to this "Action", "RPG") In thi case from -> "['Action', 'RPG']" to -> ['Action', 'RPG']
df_clean["genres"] = df_clean["genres"].str.split(",")

#Explode so each game(row) has one genre (if a game has 3 genres, Instead of one row for "The Witcher" (RPG, Adventure), you now have one row for "The Witcher" as an RPG and another row for it as an Adventure game.)
genres_df = df_clean.explode("genres")

#Strip spaces, removes accidental leading or trailing spaces (e.g., " RPG" becomes "RPG").
genres_df["genres"] = genres_df["genres"].str.strip()

#Remove "nan" strings (these were originally empty values)
genres_df = genres_df[~genres_df["genres"].isin(["","nan","None"])]
genres_df = genres_df[genres_df["genres"] != ""]

import seaborn as sns
import matplotlib.pyplot as plt

# Calculate average rating per genre
# groupby: cluster all adventure games(like this to all the games)
# ["rating"].mean(): Calculates the average rating for every game within that cluster.
# reset.index(): It converts the row labels (Index) back into a regular column and re-numbers the rows from 0 to N so your data is flat and ready to plot.
top_genres = (
    genres_df.groupby("genres")
    ["rating"].mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

# Plot with Seaborn
plt.figure(figsize=(10,6))
sns.barplot(data=top_genres, x="rating", y="genres")
plt.title("Top Rated Genres")
plt.xlabel("Average Rating (%)")
plt.ylabel("Genre")
plt.show()

**Most Played Games**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#Removing missing values if only one colums has the data(eihter the name or playtime missing)
df_play = df_clean.dropna(subset=["name","average_playtime_forever"])

#Sorting by playtime
top_games = (
    df_play
    .sort_values(by="average_playtime_forever", ascending=False)
    .head(10)
)

#Converting minutes to hours
top_games["hours_played"] = (top_games["average_playtime_forever"]/60).round(1)

#Ploting
plt.figure(figsize=(12,6))
sns.barplot(
    data=top_games,
    x="hours_played",
    y="name",
    palette="coolwarm"
)

plt.title("Top 10 Most Played Games", fontsize=16)
plt.xlabel("Total Platime (Hours)", fontsize=12)
plt.ylabel("Game", fontsize=12)

plt.show()

In [ ]:
df_clean.to_csv("steam_cleand.csv", index=False)

In [ ]:
# Keep only useful columns for your project
df_small = df_clean[[
    "name",
    "price",
    "genres",
    "rating",
    "average_playtime_forever"
]]

df_small.to_csv("steam_small.csv", index=False)